# Binaryzacja


### Cel:
- zapoznanie z segmentacją obiektów poprzez binaryzację,
- zapoznanie z binaryzacją na podstawie histogramu (globalną),
- zapoznanie z metodami automatycznego wyznaczania progu Ots'u, Kitller'a i Kapur'a,
- zapoznanie z binaryzacją lokalną (na podstawie średniej i metodą Sauvola),
- zapoznanie z binaryzacją dwuprogową,
- zadanie domowe: zapoznanie z adaptacyjną binaryzacją lokalną.

### Binaryzacja - wprowadzenie

Jednym z najważniejszych etapów podczas analizy obrazów jest segmentacja -- podział obrazu na rejony według pewnego kryterium  -- jasności, koloru, tekstury.
Najprostszą (i też najczęściej wykorzystywaną) metodą segmentacji jest **binaryzacja**.
Do jej głównych zalet zalicza się: intuicyjność, prostotę, łatwość implementacji i szybkość wykonywania.
Jest ona etapem wielu algorytmów analizy obrazów.
Pozwala na znaczną redukcję informacji w obrazie (np. dla wejściowego obrazu w skali szarości z zakresu 0-255 do 0-1).

Binaryzacja najczęściej realizowana jest poprzez progowanie.
Na przykład: dla obrazu w odcieniach szarości ustala się próg na poziomie $k$.
Wszystkie piksele o wartości (jasności) większej od $k$ zostają uznane za obiekty, a pozostałe za tło.
Oczywiście podejście takie daje się zastosować wtedy, gdy obiekty mają istotnie różną jasność od otaczającego je tła.


### Binaryzacja na podstawie histogramu

W rozdziale zostanie zademonstrowane wyznaczanie progu na podstawie "ręcznej" analizy histogramu oraz wpływ szumu i niejednorodnego oświetlenia sceny na proces binaryzacji.

1. Potrzebne w ćwiczeniu moduły są już wpisane - zwróć uwagę pod jakimi nazwami będą one widziane (plt, cv2, np).

2. Wczytaj obraz _coins.png_ w trybie odcieni szarości. Wyświetl go.
Wyznacz jego histogram (funkcja `np.histogram` lub 'cv2.calcHist') i wyświetl go.
Przy wyświetlaniu histogramu warto zwiększyć liczbę wyświetlanych wartości na osi x oraz powiększyć sam wykres (funkcje *plt.xticks(np.arange(0, 256, 20.0))* oraz *plt.rcParams["figure.figsize"] = (10,5)*.
Uwaga. Proszę powyższą funkcjonalność zaimplementować jako funkcję, gdyż przyda się w dalszej części ćwiczenia.
      


In [ ]:
import cv2
import os
import requests
from matplotlib import pyplot as plt
import numpy as np

url = 'https://raw.githubusercontent.com/vision-agh/poc_sw/master/04_Thresholding/'

fileNames = ["coins.png", "rice.png", "catalogue.png", "bart.png", "figura1.png", "figura2.png", "figura3.png", "figura4.png", "T.png"]
for fileName in fileNames:
  if not os.path.exists(fileName):
      r = requests.get(url + fileName, allow_redirects=True)
      open(fileName, 'wb').write(r.content)

In [ ]:
coin = cv2.imread("coins.png", cv2.IMREAD_GRAYSCALE)

def show_img_and_hist(image):
    fig, axes = plt.subplots(1, 2)
    fig.set_size_inches(30, 8)

    axes[0].imshow(image, "gray")
    axes[0].axis("off")

    image_hist, bins = np.histogram(image, bins=256, range=(0, 255))

    axes[1].set_xticks(np.arange(0, 256, 10))
    axes[1].plot(bins[:-1], image_hist) #bins opisuje granice przedziałów a nie środek
    axes[1].grid()

    plt.show()

show_img_and_hist(coin)

3. Wizualna analiza histogramu pozwala zauważyć dwa maksima - jedno odpowiadające poziomowi jasności tła (które w tym przypadku jest względnie jednolite - ciemnoszare) i drugie odpowiadające monetom.

Na podstawie histogramu wyznacz próg i wykonaj binaryzację:
- wykorzystaj fakt, że dla macierzy *numpy* można wykonać operację porównania wszystkich jej wartości z liczbą  - wynikiem jest macierz zawierająca wartości *True* i *False*, którą można przekonwertować metodą macierz.astype(np.int) na macierz z wartościami 1 i 0 (aczkolwiek nie jest to tu konieczne).
- wynik binaryzacji wyświetl,
- spróbuj dobrać jak najlepszy próg binaryzacji. Jako "kryterium jakości" przyjmij kształty monet - dla poprawnie dobranego progu powinny to być wypełnione koła.

Uwaga. Proszę powyższą funkcjonalność zaimplementować jako funkcję, gdyż przyda się w dalszej części ćwiczenia.

In [ ]:
def binarize_img(image, threshold, ax):
    bool_array = image >= threshold #próg górny
    binary_array = bool_array.astype(np.uint8) * 255
    ax.imshow(binary_array, 'gray')
    ax.set_title(f"Threshold: {threshold}")
    ax.axis("off")


def try_six_thresholds(img, t1, t2, t3, t4, t5, t6):
    fig, axes = plt.subplots(2, 3)
    fig.set_size_inches(40, 20)

    binarize_img(img, t1, axes[0, 0])
    binarize_img(img, t2, axes[0, 1])
    binarize_img(img, t3, axes[0, 2])
    binarize_img(img, t4, axes[1, 0])
    binarize_img(img, t5, axes[1, 1])
    binarize_img(img, t6, axes[1, 2])

try_six_thresholds(coin, 78, 80, 100, 120, 150, 180)

4. Na "stopień trudności" przeprowadzenia binaryzacji największy wpływ mają dwa czynniki:
- szum,
- niejednorodne oświetlenie.
	  
Użyj obrazy:
 - _figura1.png_ (bez zaszumienia),
 - _figura2.png_ (dodany szum Gaussowski o średniej 0 i odchyleniu standardowym 10),
 - _figura3.png_ (dodany szum Gaussowski o średniej 0 i odchyleniu standardowym 50),
 - _figura4.png_ (dodany gradient oświetlenia -- symulacja oświetlenia niejednorodnego) i wyświetl ich histogramy (wykorzystaj funkcję z poprzedniego punktu).


In [ ]:
figures = []

for i in range(1, 5):
    image = cv2.imread(f"figura{i}.png", cv2.IMREAD_GRAYSCALE)
    figures.append(image)

Spróbuj wyznaczyć progi binaryzacji na podstawie wyświetlonych histogramów.
Jak dodanie szumu wypłynęło na histogram i łatwość wyznaczania progu binaryzacji?
Czy jest to możliwe we wszystkich przypadkach?

In [ ]:
for img in figures:
    show_img_and_hist(img)

In [ ]:
thresholds = [
    [80, 90, 110, 130, 150, 170],
    [80, 90, 110, 130, 150, 170],
    [10, 50, 70, 120, 150, 180],
    [45, 47, 50, 52, 53, 55]
]

for i, img in enumerate(figures):
    th = thresholds[i]
    try_six_thresholds(img, th[0], th[1], th[2], th[3], th[4], th[5])

### Automatyczne wyznaczanie progu binaryzacji

W automatycznym systemie analizy obrazów (działanie bez nadzoru operatora) konieczne jest zastosowanie metody binaryzacji, która w sposób automatyczny wyznacza próg binaryzacji.
Oczywiście można sobie wyobrazić użycie stałego progu (np. 10), ale wtedy należy zadbać o niezmienność warunków oświetleniowych, co w niektórych zastosowaniach może być problematyczne.

#### Iteracyjne wyznaczenie progu

Jednym z najprostszych podejść jest iteracyjna procedura wyliczania progu.
Jako pierwsze przybliżenie progu ($k$) przyjmuje się średnia jasność na obrazie.
Następnie, na podstawie $k$,  dzieli się obraz na dwa podobrazy $I_0$ i  $I_1$ (dwie klasy $C_0$ i $C_1$).
Dla każdego z nich oblicza się średnią jasność: $m_0$ i $m_1$.
Jako nowy próg przyjmuje się:

$$
k_{\text{new}} = \frac{m_0 + m_1}{2} \tag{1}
$$

Procedurę kontynuuje się do momentu, aż różnica pomiędzy dwoma kolejnymi progami będzie mniejsza niż zadana wartość.


**Zadanie: zaimplementować opisany powyżej algorytm.**


Jak można zauważyć, do poprawnego działania metody potrzebne będzie obliczanie średniej jasności, również dla pewnych podobrazów.
Wykorzystamy do tego znormalizowanych histogram:

$$
p_i = \frac{n_i}{N}, \quad \sum_{i=0}^{L-1} p_i = 1 \tag{2}
$$

gdzie: $n_i$ liczba pikseli o jasności $i$ ($i = 0,1, ... L-1$) - histogram, $L$ - liczba poziomów jasności, $N$ - liczba pikseli na obrazie ($N = n_0 + n_1 + ... + n_{L-1}$).

Jeśli podzielimy obraz na dwie klasy $C_0$ i $C_1$ (tło i obiekty albo obiekty i tło) z progiem podziału oznaczonym jako $k$, to do klasy $C_0$ należeć będą piksele o poziomach $[0,k]$, a do klasy $C_1$ piksele o poziomach $[k+1,L-1]$.

Wtedy prawdopodobieństwo, że piksel należy do klasy $C_0$ wynosi:

$$
P_0(k) = \sum_{i=0}^{k} p_i \tag{3}
$$

Podobnie prawdopodobieństwo, że należy do klasy $C_1$ wynosi:

$$
P_1(k) = \sum_{i=k+1}^{L-1} p_i = 1 - P_0(k) \tag{4}
$$

Średnią jasność pikseli należących do klasy $C_0$ można wyznaczyć na podstawie:

$$
m_0(k) = \sum_{i=0}^{k} iP(i|C_0) \tag{5}
$$

gdzie: $|$ oznacza prawdopodobieństwo warunkowe, a wyraz $P(i|C_0)$ - prawdopodobieństwo dla wartości $i$ pod warunkiem, że $i$ należy do klasy $C_0$.
Równanie to jest szczególnym przypadkiem wykorzystania momentów statystycznych do wyliczania pewnych parametrów statystycznych - w tym przypadku średniej.

Wykorzystując regułę Bayesa:

$$
P(A|B) = \frac{P(B|A)P(A)}{P(B)} \tag{6}
$$

możemy zapisać:

$$
m_0(k) = \sum_{i=0}^{k} i \frac{P(C_0|i) P(i)}{P(C_0)} \tag{7}
$$

Wyraz $P(C_0|i) = 1$, gdyż z założenia rozpatrujemy tylko piksele należące do klasy $C_0$.
Wyraz $P(i)$ stanowi $i$-ty element znormalizowanego histogramu tj. $P(i) = p_i$, a $P(C_0)$ to prawdopodobieństwo przynależności do klasy $C_0$ określone wcześniej $P(C_0) = P_0(k)$.
Ostatecznie możemy więc zapisać:

$$
m_0(k) = \frac{1}{P_0(k)} \sum_{i=0}^{k} i p_i \tag{8}
$$

Na podstawie analogicznych rozważań można wyprowadzić wzór na średnią jasności pikseli należących do klasy $C_1$:

$$
m_1(k) = \frac{1}{P_1(k)} \sum_{i=k+1}^{L-1} i p_i \tag{9}
$$

Średnia jasność całego obrazu dana jest zależnością:

$$
m_G = \sum_{i=0}^{L-1} i p_i \tag{10}
$$


1. Wczytaj obraz _coins.png_. Wyświetl go.

2. Wylicz histogram i histogram skumulowany (funkcja `np.cumsum`).
   Na podstawie zależności (10) wylicz średnią - pierwszy próg podziału $k$.
   Uwagi:
   - przed dalszymi obliczeniami dobrze jest usunąć zbędny wymiar tablicy z histogramem - polecenie `np.squeeze`
   - $p_i$ to nasz znormalizowany histogram, a wartości od $0$ do $255$ można wygenerować poleceniem `np.arange(256)`
   - zmiast pętli `for` można wykorzystać iloczyn skalarny dwóch wektorów tj. `np.dot`

3.  W nieskończonej pętli `while` wykonaj następujące kroki:
- oblicz średnią $m_0$ -- zależność (8)
    - dla $P_0$ wystarczy wykorzystać odpowiednią wartość znormalizowanego histogramu skumulowanego, dla pozostałej części wyrażenia podobne rozwiązanie jak dla pierwszej średniej
- oblicz średnią $m_1$ -- zależność (9)
- oblicz nowy próg $k_{\text{new}}$ -- zależność (1)
- oblicz moduł z różnicy pomiędzy $k_{\text{new}}$, a $k$ i sprawdź czy jest mniejszy od progu (np. 1)
- jeśli tak to zakończ obliczenia (`break`), jeśli nie to przypisz $k = k_{\text{new}}` i kontynuuj obliczenia
- wyświetl próg oraz wynik binaryzacji

4. Sprawdz jak metoda dziala na obrazach _figura1.png_ do _figura4.png_.

In [ ]:
def iter_threshold(image, epsilon, inital_threshold):
    mean = np.mean(image)
    img_1 = image[image < mean]
    img_2 = image[image >= mean]

    mean_1 = np.mean(img_1)
    mean_2 = np.mean(img_2)

    new_threshold = (mean_1 + mean_2)/2

    if abs(inital_threshold - new_threshold) <= epsilon:
        return new_threshold
    else:
        return iter_threshold(image, epsilon, new_threshold)

fig, axes = plt.subplots(1, 4)
fig.set_size_inches(40, 20)

def normalized_hist(img):
    hist = cv2.calcHist([img], [0], None, [256], [0, 255])
    return hist.squeeze()/hist.cumsum()[-1]

def calc_inital_threshold(img):
    p_i = normalized_hist(img)
    brightness_values = np.arange(256)
    return np.dot(p_i, brightness_values)

for i, img in enumerate(figures):
    initial_treshold = calc_inital_threshold(img)
    threshold = iter_threshold(img, 1, int(initial_treshold))
    binarize_img(img, threshold, axes[i])

In [ ]:
def bayes_treshold(image, inital_threshold, p_i):
    inital_threshold = int(inital_threshold)
    p_0_k = sum(p_i[:inital_threshold + 1])
    p_1_k = 1 - p_0_k

    m_0_k = (1/p_0_k) * np.dot(brightness_values[:inital_threshold + 1], p_i[: inital_threshold + 1])
    m_1_k = (1/p_1_k) * np.dot(brightness_values[inital_threshold + 1:], p_i[inital_threshold + 1:])

    new_threshold = (m_0_k + m_1_k)/2

    if abs(inital_threshold - new_threshold) <= 1:
        return new_threshold
    else:
        return bayes_treshold(image, new_threshold, p_i)

fig, axes = plt.subplots(1, 4)
fig.set_size_inches(40, 20)
brightness_values = np.arange(256)

for i, img in enumerate(figures):
    p_i = normalized_hist(img)
    initial_treshold = calc_inital_threshold(img)
    threshold = bayes_treshold(img, int(initial_treshold), p_i)
    binarize_img(img, threshold, axes[i])

#### Metoda Otsu

Jednym z częściej wykorzystywanych algorytmów wyznaczania progu jest metoda zaproponowana w roku 1979 przez Nobuyuki Otsu w artykule pt. "A Threshold Selection Method from Gray-Level Histograms" (można odszukać na IEEE Xplore).
W algorytmie zakłada się, że obraz zawiera piksele należące do dwóch klas (obiektów i tła) tj. histogram obrazu jest bi-modalny (ma dwa maksima).
Próg podziału obliczany jest tak, aby wariancja międzyklasowa była maksymalna.
W tym sensie metodę Otsu można nazwać optymalną.

Wprowadźmy teraz wskaźnik "jakości" wybranego progu podziału $k$, który będziemy optymalizować.
W algorytmie Otsu jest to:

$$
\eta(k) = \frac{\sigma^2_B(k)}{\sigma^2_G} \tag{11}
$$

gdzie:  $\sigma^2_G$ - wariancja globalna, która może zostać obliczona na podstawie momentów statystycznych jako:

$$
\sigma^2_G =  \sum_{i=0}^{L-1} (i - m_G)^2 p_i \tag{12}
$$

a $\sigma^2_B$ jest wariancją międzyklasową, która jest zdefiniowana jako:

$$
\sigma^2_B(k) =  P_0(k)(m_0(k) - m_G)^2 + P_1(k)(m_1(k) - m_G)^2 \tag{13}
$$

Równianie to można również przekształcić do:

$$
\sigma^2_B(k) =  P_0(k)P_1(k)(m_0(k) - m_1(k))^2 = \frac{(m_G P_0(k) - m(k) )^2}{P_0(k)(1-P_0(k))} \tag{14}
$$

gdzie:

$$
m(k) = \sum_{i=0}^{k} i p_i \tag{15}
$$

Taki zapis pozwala przyspieszyć obliczenia.
Wartość $m_G$ wyznaczana jest jednokrotnie, a zachodzi tylko potrzeba obliczania $m(k)$ i $P_0(k)$ w każdej iteracji.
Warto też zwrócić uwagę, że równanie ma sens dla $P_0 > 0$.

Warto zauważyć, że z postaci równania (14) wynika, że im większa odległość pomiędzy średnimi $m_0$ i $m_1$ tym wartość wariancji międzyklasowej jest większa.
Pokazuje to, że przyjęty współczynnik może być podstawą do separacji dwóch klas - im jego wartość jest większa, tym lepsze rozdzielenie.
Dodatkowo, z równania (11) wynika, że $\eta(k)$ zależy tylko od wariancji międzyklasowej $\sigma^2_B(k)$, gdyż wariancja globalna $\sigma^2_G$ jest stała.
Zatem w procesie optymalizacji należy dążyć do maksymalizacji wskaźnika $\eta$.

Należy też pamiętać, że współczynnik jest poprawnie określony tylko dla wartości $\sigma^2_G > 0$.
Przy czym, wartość 0 może on przyjąć tylko dla obrazu o jednym poziomie szarości - w takim przypadku trudno mówić o podziale pikseli na dwie klasy (skoro występuje tylko jedna).

Ostatecznie optymalny próg binaryzacji $\bar{k}$ wyliczamy na podstawie zależności:

$$
\sigma^2_B(\bar{k}) = \max\limits_{k \in[0,L-1]} {\sigma^2_B(k) } \tag{16}
$$

Uwagi:
- może się zdarzyć, że znajdziemy więcej niż jedno maksimum tj. więcej wartości $\bar{k}$.
  W takim przypadku zwykle zakłada się, że próg będzie średnią otrzymanych wartości.
- liczby $P_0(\bar{k})$ i $P_1(\bar{k})$ odpowiadają powierzchni zajmowanej przez obiekty klas $C_0$ i $C_1$.
- liczby $m_0(\bar{k})$ i $m_1(\bar{k})$ odpowiadają średniej jasności obiektów klas $C_0$ i $C_1$.
- wartość parametru $\eta(\bar{k})$ określa "jakość" wyznaczonego progu -- im większa tym lepiej.

Zadanie: wykorzystując podane powyżej informacje należy zaimplementować metodę wyznaczania progu binaryzacji zaproponowaną przez Otsu.

1. Wczytaj obraz _coins.png_.
      Wyświetl go.

2. Wyznacz jego histogram znormalizowany oraz oblicz średnią jasność (można do tego wykorzystać histogram) - kod zbliżony do stworzonego wcześniej.

3. Zdefiniuj 256-elementowy wektor na współczynniki $\sigma_B^2$ (funkcja `np.zeros`).

4. W pętli po możliwych wartościach progu binaryzacji wyznacz wartość $\sigma_B^2(k)$ na podstawie zależności (14).
      Uwagi:
      - wcześniejszego liczenia wartości $P_0(k)$ i $m(k)$ można uniknąć inkrementując wartośc $P_0, m$  w każdej iteracji.
      - należy pamiętać, że równanie ma sens tylko dla $0 < P_0(k) < 1$. <br>

5. Wyświetl przebieg $\sigma_B^2(k)$.
      Wykorzystaj funkcję `plt.plot`.

6. Wyznacz wartość $\bar{k}$ dla której współczynnik $\sigma_B^2$ jest maksymalny.
	  Można to zrobić poprzez dodanie instrukcji w pętli (rozwiązanie bardziej eleganckie) lub wykorzystując funkcję `max` (rozwiązanie dla leniwych).
	  Uwaga. Proszę pominąć obsługę przypadków niejednoznacznego maksimum.

7. Zbinaryzuj obraz wykorzystując otrzymany próg.
      Porównaj wyniki z rezultatem binaryzacji "ręcznej".

8. W OpenCV dostępna jest implementacja metody Otsu - funkcja `cv2.threshold` z parametrem `cv2.THRESH_OTSU`.
      Funkcja zwraca zbinaryzowany obraz oraz próg.
      Wykonaj binaryzację obrazu _coins.png_ metodą Otsu.
      Porównaj wyniki z własną implementacją - powinno wyjść tak samo (tzn. taki sam próg).

9. Przeprowadź eksperyment również na obrazie _rice.png_ i _catalogue.png_

In [ ]:
rice = cv2.imread("rice.png", cv2.IMREAD_GRAYSCALE)
catalogue = cv2.imread("catalogue.png", cv2.IMREAD_GRAYSCALE)
figures.append(rice)
figures.append(catalogue)

figures = figures[:6]

In [ ]:

n = len(figures)

chart, axes_chart = plt.subplots(1, n)
chart.set_size_inches(40, 10)

fig, axes = plt.subplots(1, n)
fig.set_size_inches(40, 20)

manual, axes_m = plt.subplots(1, n)
manual.set_size_inches(40, 20)

cv, axes_cv = plt.subplots(1, n)
cv.set_size_inches(40, 20)

cv_2, axes_cv_2 = plt.subplots(1, n)
cv_2.set_size_inches(40, 20)

def otsu(i, img):
    p_i = normalized_hist(img)
    brightness_values = np.arange(256)

    m_G = np.dot(brightness_values, p_i)

    sigma_B2 = np.zeros(256)

    P0 = 0
    m = 0

    for k in range(256):
        P0 += p_i[k]
        m += k * p_i[k]

        if P0 == 0 or P0 == 1:
            sigma_B2[k] = 0
        else:
            sigma_B2[k] = ((m_G * P0 - m)**2) / (P0 * (1 - P0))

    axes_chart[i].plot(sigma_B2)
    axes_chart[i].set_title("Wariancja międzyklasowa sigma_B2(k)")
    axes_chart[i].set_xlabel("Próg k")
    axes_chart[i].set_ylabel("sigma_B2")
    axes_chart[i].grid(True)

    k_opt = np.argmax(sigma_B2)
    print(f"Optymalny próg Otsu: {k_opt}")

    binarize_img(img, k_opt, axes[i])

for i, img in enumerate(figures):
    otsu(i, img)
    initial_threshold = calc_inital_threshold(img)
    p_i = normalized_hist(img)
    threshold = bayes_treshold(img, initial_threshold, p_i)
    binarize_img(img, threshold, axes_m[i])
    cv2_threshold, otsu_img = cv2.threshold(img, 0, 255, cv2.THRESH_OTSU)
    print(f"Z funkcji z cv2: {cv2_threshold}")
    binarize_img(img, cv2_threshold, axes_cv[i])
    axes_cv_2[i].imshow(otsu_img, 'gray')
    axes_cv_2[i].set_title(f"Threshold: {cv2_threshold}")
    axes_cv_2[i].axis("off")

### Binaryzacja lokalna


Analiza wyników binaryzacji dla obrazów _rice.png_ i _catalogue.png_ pokazuje, że globalna binaryzacja nie najlepiej działa dla obrazów o niejednorodnym oświetleniu.
Dla obu obrazów trudno również wyznaczyć odpowiedni próg "ręcznie".

Metodą, która pozwala poprawić wyniki binaryzacji, jest binaryzacja lokalna (niekiedy zwana adaptacyjną).
W jednym z wariantów polega ona na wyznaczeniu progu osobno dla każdego piksela na podstawie jego otoczenia (tj. własności jego kontekstu, okna).

1. Dla uproszczenia zakładamy, że obraz ma rozmiar $256 \times 256$ pikseli. Przyjmijmy okno analizy o rozmiarze 15 pikseli.

2. Najprostsza wersja binaryzacji lokalnej zakłada, że próg binaryzacji dla danego okna to średnia z pikseli w tym oknie.

3. Wczytaj obraz _rice.png_. Rozmiar obrazka (`X,Y`) można uzyskać stosując taką składnię: `(X, Y) = obraz.shape`.

4. Podstawą algorytmu są dwie pętle `for` iterujące po pikselach obrazka:


	for j in range(W/2, Y-W/2):
	    for i in range(W/2, X-W/2):

5. Wewnątrz pętli należy dla każdego piksela wyciąć jego otoczenie o rozmiarze `W` (operator `:`), wyznaczyć z niego średnią (metoda `mean`) i na jej podstawie dokonać binaryzacji.

6. Wyświetl obrazy oryginalny i zbinaryzowany.

7. Zaobserwuj rezultaty działania metody dla obrazów _rice.png_ i _catalogue.png_.
      Poeksperymentuj z rozmiarem okna (proszę nie przesadzać z rozmiarem, gdyż istotnie wpływa on na czas obliczeń).
      Jaka jest podstawowa wada zaimplementowanej metody? (pomijając złożoność obliczeniową).
      Proszę się zastanowić jakie jest źródło błędów.

In [ ]:

def bin_local(img, W):
    X, Y = img.shape
    binary_local = np.zeros_like(img)
    half_W = W//2

    for j in range(half_W, Y-half_W):
        for i in range(half_W, X-half_W):
            window = img[i - half_W : i + half_W + 1, j - half_W : j + half_W + 1]

            local_mean = np.mean(window)

            if img[i, j] >= local_mean:
                binary_local[i, j] = 255
            else:
                binary_local[i, j] = 0
                
    plt.figure(figsize=(7, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap='gray')
    plt.title("Oryginalny obraz (rice.png)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(binary_local, cmap='gray')
    plt.title(f"Lokalna binaryzacja (okno {W}x{W})")
    plt.axis('off')

    plt.show()

for img in figures:
    bin_local(img, 15)

8. Jakość działania binaryzacji lokalnej można poprawić wyznaczając próg za pomocą metody Sauvol'i i Pietikainen'a zaproponowanej w artykule *Adaptive document image binarization*.
Wykorzystuje ona, oprócz średniej, informację o odchyleniu standardowym w danym oknie.
Próg binaryzacji wyznaczany jest na podstawie zależności:

$$
T = \text{srednia}\,\left[\,1 \pm k\left(\frac{\text{odchStd}}{R} - 1\right)\,\right]
$$

gdzie: $k$ i $R$ to parametry ($R$ zwykle $128$, a $k$ na początek przyjmij $0.15$), $srednia$ i $odchStd$ to odpowiednio średnia i odchylenie standardowe wyliczone w oknie.

9. Zaimplementuj algorytm Sauvoli - wykorzystaj do wyznaczenia średniej i odchylenia metody `mean()` oraz `std()` liczone dla wycinka (podobnie jak średnia w poprzedniej metodzie).
      
10. Uruchom metodę (uwaga - czas obliczeń nie jest krótki). Przeanalizuj wyniki. Zwróć uwagę, że dodanie informacji o odchyleniu standardowym powinno *poprawić* wyniki binaryzacji.
      Jeżeli dzieje się inaczej, to najprawdopodobniej implementacja zawiera błąd.
     
11. Zastanów się nad znaczeniem symbolu $\pm$ we wzorze na próg.
      Kiedy należy zastosować znak +, a kiedy -.

12. Porównaj jakość binaryzacji lokalnej metodą Sauvol'i i z progiem na podstawie średniej.
      Poeksperymentuj z rozmiarem okna i parametrem k (dla obrazów _rice.png_ i _catalogue.png_).

In [ ]:
def bin_sauvola(img, W, k=0.15, R=128):
    X, Y = img.shape
    binary_sauvola = np.zeros_like(img)
    half_W = W // 2

    for j in range(half_W, Y - half_W):
        for i in range(half_W, X - half_W):
            window = img[i - half_W : i + half_W + 1, j - half_W : j + half_W + 1]
            mean = np.mean(window)
            std = np.std(window)

            T = mean * (1 + k * (std / R - 1))

            if img[i, j] >= T:
                binary_sauvola[i, j] = 255
            else:
                binary_sauvola[i, j] = 0

    plt.figure(figsize=(7, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap='gray')
    plt.title("Oryginał")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(binary_sauvola, cmap='gray')
    plt.title(f"Sauvola (okno {W}x{W}, k={k})")
    plt.axis('off')
    plt.show()

for img in figures:
    bin_sauvola(img, 15, k=0.15)

### Binaryzacja dwuprogowa

Binaryzację można przeprowadzić wykorzystując więciej niż jedn próg.
Przykładem jest binaryzacja dwuprogowa - wybieramy w ten sposób przedział jasności (piksele w nim zawarte klasyfikujemy jako obiekty).

1. Wczytaj obraz _bart.png_.
Wyświetl go, wyznacz i wyświetl jego histogram.
Oceń, który fragment histogramu odpowiada kolorowi skóry Barta Simpsona.<br>
**UWAGA - Aby odczytać wartości pikseli można zapisać obrazek na dysku (`cv2.imwrite('Nazwa.png', Image)`), a następnie odczytać wartościa programem do edycji obrazów, np. *paint*.**<br>

In [ ]:
bart = cv2.imread("bart.png", cv2.IMREAD_GRAYSCALE)

hist = cv2.calcHist([bart], [0], None, [256], [0,256])
show_img_and_hist(bart)

cv2.imwrite('bart_grayscale.png', img)

In [ ]:
low = 190
high = 210

binary_bart = np.zeros_like(bart)
binary_bart[(bart >= low) & (bart <= high)] = 255
plt.imshow(binary_bart, cmap='gray')
plt.title(f"Binaryzacja dwuprogowa: {low}-{high}")
plt.axis('off')
plt.show()


2. Przeprowadź segmentację na podstawie koloru skóry (binaryzację dwuprogową).
      Wykorzystaj przekształcenie obrazów z wartościami True, False na wartości 1,0 i mnożenie obrazów. Wyświetl wynik.
      

In [ ]:
mask = (bart >= low) & (bart <= high)

mask_int = mask.astype(np.uint8)

segmented = bart * mask_int

plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(bart, cmap='gray')
plt.title("Odcienie szarości")
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(segmented, cmap='gray')
plt.title("Dwuprogowa binaryzacja")
plt.axis('off')
plt.show()